In [2]:
# import pandas as pd
# import matplotlib.pyplot as plt

# import seaborn as sns
import requests
import json
from datetime import datetime, timedelta

In [3]:
# The NWS API requires a user agent header with contact information
headers = {
    'User-Agent': '(MeriWeather, zachary.a.mays@gmail.com)',
    'Accept': 'application/geo+json'
}

In [4]:
# Step 1: Find a weather station near your location
# You can use latitude/longitude coordinates to find nearby stations
def find_stations(lat, lon):
    point_url = f"https://api.weather.gov/points/{lat},{lon}"
    response = requests.get(point_url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        # Get the URL for the nearest observation stations
        stations_url = data['properties']['observationStations']
        
        # Get the list of stations
        stations_response = requests.get(stations_url, headers=headers)
        if stations_response.status_code == 200:
            return stations_response.json()['features']
    
    return None

# Step 2: Get historical observations from a specific station
def get_historical_observations(station_id, start_date, end_date):
    url = f"https://api.weather.gov/stations/{station_id}/observations"
    
    # Format dates as ISO 8601
    params = {
        'start': start_date.isoformat() + "Z",
        'end': end_date.isoformat() + "Z"
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [7]:
# Example usage
if __name__ == "__main__":
    # Coordinates for Chelsea, MI
    lat, lon = 42.3181, 84.0205
    
    # Find nearby stations
    stations = find_stations(lat, lon)
    print(stations)
    
    if stations and len(stations) > 0:
        # Use the first station found
        station_id = stations[0]['properties']['stationIdentifier']
        print(f"Using station: {station_id}")
        
        # Get data for the past 7 days
        end_date = datetime.now(datetime.timezone.utc)
        start_date = end_date - timedelta(days=7)
        
        observations = get_historical_observations(station_id, start_date, end_date)
        
        if observations:
            # Process and display the observations
            for obs in observations['features']:
                timestamp = obs['properties']['timestamp']
                temp = obs['properties']['temperature']['value']
                
                if temp is not None:
                    temp_f = (temp * 9/5) + 32  # Convert from C to F
                    print(f"Time: {timestamp}, Temperature: {temp_f:.1f}°F")
else:
    print("No stations found.")

None
